<a href="https://colab.research.google.com/github/Marzuq-sci/Africa-Quantum-Revolution/blob/main/VNA_Large_scale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Global Microgrid Optimization with VNA and CGS

**Author:** Marzuq Yussif Etsie Adam et al.
**Paper:** QUBO Models for Energy Planning: Quantum-Inspired Microgrid Optimization through Variational Neural Annealing

This notebook combines:
* **Symbolic computing** (SymPy) to derive exact QUBO formulations and penalty terms.
* **Zeta regularization** to handle decoherence and divergent sums.
* **Analytic continuity** to justify the annealing schedule (Wick rotation).
* **Variational Neural Annealing (VNA)** with an autoregressive RNN to solve the QUBO.

We use real global city data (Natural Earth) and a Cartesian Grid System (CGS) to enforce geographic dispersion.

NameError: name 'json' is not defined

# Global Microgrid Optimization with VNA and CGS

**Author:** Marzuq Yussif Etsie Adam et al.
**Paper:** QUBO Models for Energy Planning: Quantum-Inspired Microgrid Optimization through Variational Neural Annealing

This notebook combines:
* **Symbolic computing** (SymPy) to derive exact QUBO formulations and penalty terms.
* **Zeta regularization** to handle decoherence and divergent sums.
* **Analytic continuity** to justify the annealing schedule (Wick rotation).
* **Variational Neural Annealing (VNA)** with an autoregressive RNN to solve the QUBO.

We use real global city data (Natural Earth) and a Cartesian Grid System (CGS) to enforce geographic dispersion.

In [5]:
!apt-get install -y libspatialindex-dev
!pip install geopandas pyproj rtree shapely pandas numpy matplotlib torch sympy

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libspatialindex-dev is already the newest version (1.9.3-2).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [6]:
import geopandas as gpd
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import random
import sympy as sp
from sympy import symbols, Matrix, simplify, init_printing
init_printing()

In [7]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [8]:
This cell was attempting to load data that is no longer available. Data loading and preprocessing is now handled by the synthetic data generation in the next cells.

SyntaxError: invalid syntax (315286500.py, line 1)

In [9]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import torch
from shapely.geometry import Point

# 1. Generate Synthetic Global Data
print("Generating synthetic global city data...")
n_synth = 5000
np.random.seed(SEED)
lons = np.random.uniform(-180, 180, n_synth)
lats = np.random.uniform(-60, 80, n_synth)
pops = np.random.exponential(scale=100000, size=n_synth) + 5000

gdf = gpd.GeoDataFrame({
    'NAME': [f'City_{i}' for i in range(n_synth)],
    'SOV0NAME': ['Synthetic'] * n_synth,
    'POP_MAX': pops,
    'geometry': [Point(x, y) for x, y in zip(lons, lats)]
}, crs='EPSG:4326')

# 2. Process Data
gdf['lon'] = gdf.geometry.x
gdf['lat'] = gdf.geometry.y
df = pd.DataFrame({
    'city': gdf['NAME'],
    'lat': gdf['lat'],
    'lon': gdf['lon'],
    'pop': gdf['POP_MAX']
})
df['cost'] = 100 * df['pop'] + np.random.normal(0, 500000, len(df))
df['cost'] = df['cost'].clip(0, None)
df['demand'] = df['pop'] * 1.0

# Normalize
df['cost_norm'] = (df['cost'] - df['cost'].min()) / (df['cost'].max() - df['cost'].min())
df['pop_norm'] = (df['pop'] - df['pop'].min()) / (df['pop'].max() - df['pop'].min())
df['demand_norm'] = (df['demand'] - df['demand'].min()) / (df['demand'].max() - df['demand'].min())

# Grid Assignment
df['cell_id'] = ((df['lat'] // 2.0).astype(int) * 1000) + ((df['lon'] // 2.0).astype(int) + 90)
N = len(df)

print(f"Data Ready: {N} cities processed. Starting VNA Training...")

Generating synthetic global city data...
Data Ready: 5000 cities processed. Starting VNA Training...


In [20]:
# 4. Symbolic QUBO Derivation
C_i, P_i, E_i = symbols('C_i P_i E_i', real=True)
B, M = symbols('B M', real=True)
lambda_B, lambda_P, lambda_G = symbols('lambda_B lambda_P lambda_G', positive=True)
alpha1, alpha2, alpha3 = symbols('alpha1 alpha2 alpha3', positive=True)

budget_penalty = lambda_B * (C_i - B)**2
pop_penalty = lambda_P * (P_i - M)**2
objective = alpha1*C_i - alpha2*P_i - alpha3*E_i
H_diag = objective + budget_penalty + pop_penalty

C_j, P_j = symbols('C_j P_j')
H_off = 2*lambda_B*(C_i - B)*(C_j - B) + 2*lambda_P*(P_i - M)*(P_j - M) + lambda_G

print("Diagonal term symbolic derivation complete.")

Diagonal term symbolic derivation complete.


In [19]:
# 5. Zeta Regularization & 6. Analytic Continuity
from sympy import zeta, limit
s = sp.Symbol('s', real=True)
mu = sp.Symbol('mu', positive=True)

print(f"Regularized sum zeta(-1): {zeta(-1).evalf()}")

t, tau, beta = sp.symbols('t tau beta', real=True)
print("Wick rotation mapping: t -> -i*tau => beta = tau")

Regularized sum zeta(-1): -0.0833333333333333
Wick rotation mapping: t -> -i*tau => beta = tau


In [21]:
# Hyperparameters (start with very small penalties)
alpha1, alpha2, alpha3 = 1.0, 10.0, 5.0
lambda_B, lambda_P, lambda_G = 0.1, 0.1, 0.05   # reduced from 100,100,50
budget_raw = 50_000_000_000
target_pop_raw = 500_000_000

# Normalize budget and target using the same scaling as features
cost_min, cost_max = df['cost'].min(), df['cost'].max()
pop_min, pop_max = df['pop'].min(), df['pop'].max()
budget_norm = (budget_raw - cost_min) / (cost_max - cost_min)
target_pop_norm = (target_pop_raw - pop_min) / (pop_max - pop_min)

# (Optional) Clip to [0,1] – uncomment if desired
# budget_norm = np.clip(budget_norm, 0, 1)
# target_pop_norm = np.clip(target_pop_norm, 0, 1)

C = torch.tensor(df['cost_norm'].values, dtype=torch.float32)
P = torch.tensor(df['pop_norm'].values, dtype=torch.float32)
E = torch.tensor(df['demand_norm'].values, dtype=torch.float32)
cells = torch.tensor(df['cell_id'].values, dtype=torch.long)
N = len(df)

# Diagonal
Q_diag = alpha1*C - alpha2*P - alpha3*E \
         + lambda_B*(C**2 - 2*budget_norm*C) \
         + lambda_P*(P**2 - 2*target_pop_norm*P)

# Sparse off-diagonals (same cell penalty)
off_indices = []
off_vals = []
cell_groups = df.groupby('cell_id').indices
for cid, idxs in cell_groups.items():
    if len(idxs) > 1:
        for i in range(len(idxs)):
            for j in range(i+1, len(idxs)):
                u, v = idxs[i], idxs[j]
                val = 2*lambda_B*C[u]*C[v] + 2*lambda_P*P[u]*P[v] + lambda_G
                off_indices.append((u, v))
                off_vals.append(val)

if off_indices:
    off_indices = torch.tensor(off_indices, dtype=torch.long).t()
    off_vals = torch.tensor(off_vals, dtype=torch.float32)
else:
    off_indices = torch.empty(2, 0, dtype=torch.long)
    off_vals = torch.empty(0, dtype=torch.float32)

def qubo_energy(x):
    e = torch.sum(x * Q_diag, dim=1)
    if off_indices.numel() > 0:
        x_i = x[:, off_indices[0]]
        x_j = x[:, off_indices[1]]
        e += torch.sum(x_i * x_j * off_vals, dim=1)
    return e

# Test with a random configuration
test_x = torch.randint(0, 2, (1, N), dtype=torch.float32)
print(f"Test energy (random config): {qubo_energy(test_x).item():.2f}")

Test energy (random config): -75117.10


In [ ]:
# =============================================================================
# Training (with reduced penalties)
# =============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoregressiveRNN(input_dim=2, hidden_dim=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_steps = 3000
batch_size = 200
T0 = 10.0
gamma = 0.999
T = T0

history = {'step': [], 'energy': [], 'entropy': [], 'free': []}

for step in range(num_steps):
    optimizer.zero_grad()
    x = model.sample(N, batch_size, device)
    energy = qubo_energy(x)

    logits = model(x).squeeze(-1)
    probs = torch.sigmoid(logits)
    log_prob = x * torch.log(probs + 1e-10) + (1-x) * torch.log(1-probs + 1e-10)
    log_prob_sum = torch.sum(log_prob, dim=1)

    free = energy + T * log_prob_sum
    loss = torch.mean(free)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    T = T0 * (gamma ** (step+1))

    if step % 100 == 0:
        with torch.no_grad():
            avg_energy = energy.mean().item()
            avg_entropy = -log_prob_sum.mean().item()
            avg_free = free.mean().item()
            history['step'].append(step)
            history['energy'].append(avg_energy)
            history['entropy'].append(avg_entropy)
            history['free'].append(avg_free)
            print(f"Step {step:4d}, T={T:.4f}, Energy={avg_energy:.2f}, Entropy={avg_entropy:.2f}, Free={avg_free:.2f}")

Step    0, T=9.9900, Energy=-75029.66, Entropy=3465.48, Free=-109684.43
